In [1]:
import pandas as pd 
import numpy as np 
import holidays

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)

In [2]:
df = pd.read_parquet('../retail_data/sales_data.parquet', engine = 'pyarrow')

In [3]:
df.head()

,date,region,city,store,store_id,product,brand,category,price,discount,final_price,inventory,demand,sales_qty,sales,in_mall,store_size_sqft,local_competitor_count,foot_traffic_index,parking_available,store_age_years,online_pickup_point,population_density,gdp_per_capita,mall_count,...,weekend_boost,festival_boost,holiday_boost,stockout_flag,lost_sales,is_promo_campaign,promo_type,marketing_spend,temperature,rainfall_mm,humidity_pct,weather_condition,weather_demand_mod,is_month_start,is_month_end,salary_week_flag,supplier_lead_time_days,reorder_point,area_type,premium_store_flag,student_area_flag,office_area_flag,avg_basket_size,competitor_price_index,festival_intensity_score
0,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Samsung Electronics 1,Samsung,Electronics,32452,0,32452.0,14,16,14,454328.0,0,3592,4,52,1,11,0,20000,210000,24,...,1.0,1.0,1.0,1,2,0,,0,27.6,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0394,0
1,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Samsung Electronics 2,Samsung,Electronics,21140,0,21140.0,16,17,16,338240.0,0,3592,4,52,1,11,0,20000,210000,24,...,1.0,1.0,1.0,1,1,0,,0,27.6,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0226,0
2,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Samsung Electronics 3,Samsung,Electronics,20557,20,16445.6,14,13,13,213792.8,0,3592,4,52,1,11,0,20000,210000,24,...,1.0,1.0,1.0,0,0,0,,0,27.6,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0276,0
3,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Sony Electronics 2,Sony,Electronics,49535,0,49535.0,28,20,20,990700.0,0,3592,4,52,1,11,0,20000,210000,24,...,1.0,1.0,1.0,0,0,0,,0,27.6,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0190,0
4,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,LG Electronics 1,LG,Electronics,59410,15,50498.5,30,25,25,1262462.5,0,3592,4,52,1,11,0,20000,210000,24,...,1.0,1.0,1.0,0,0,0,,0,27.6,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0216,0


In [4]:
df[['price', 'discount', 'sales']] = df[['price', 'discount', 'sales']].round(2)

In [5]:
df['date'] = pd.to_datetime(df['date'], format = "%Y-%m-%d")

In [6]:
df = df.assign(
    year = df['date'].dt.year,
    month = df['date'].dt.month,
    week = df['date'].dt.isocalendar().week,
    day = df['date'].dt.day,
    quarter = df['date'].dt.quarter,
    day_of_week = df['date'].dt.dayofweek,
    is_weekend = df['date'].dt.dayofweek.isin([5, 6]).astype(int),
    is_month_start = df['date'].dt.is_month_start.astype(int),
    is_month_end = df['date'].dt.is_month_end.astype(int),
    is_year_end = df['date'].dt.is_year_end.astype(int),
    is_year_start = df['date'].dt.is_year_start.astype(int),
    is_quarter_end = df['date'].dt.is_quarter_end.astype(int)
)

In [7]:
df.head()

,date,region,city,store,store_id,product,brand,category,price,discount,final_price,inventory,demand,sales_qty,sales,in_mall,store_size_sqft,local_competitor_count,foot_traffic_index,parking_available,store_age_years,online_pickup_point,population_density,gdp_per_capita,mall_count,...,rainfall_mm,humidity_pct,weather_condition,weather_demand_mod,is_month_start,is_month_end,salary_week_flag,supplier_lead_time_days,reorder_point,area_type,premium_store_flag,student_area_flag,office_area_flag,avg_basket_size,competitor_price_index,festival_intensity_score,year,month,week,day,quarter,day_of_week,is_year_end,is_year_start,is_quarter_end
0,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Samsung Electronics 1,Samsung,Electronics,32452,0,32452.0,14,16,14,454328.0,0,3592,4,52,1,11,0,20000,210000,24,...,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0394,0,2016,1,53,1,1,4,0,1,0
1,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Samsung Electronics 2,Samsung,Electronics,21140,0,21140.0,16,17,16,338240.0,0,3592,4,52,1,11,0,20000,210000,24,...,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0226,0,2016,1,53,1,1,4,0,1,0
2,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Samsung Electronics 3,Samsung,Electronics,20557,20,16445.6,14,13,13,213792.8,0,3592,4,52,1,11,0,20000,210000,24,...,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0276,0,2016,1,53,1,1,4,0,1,0
3,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,Sony Electronics 2,Sony,Electronics,49535,0,49535.0,28,20,20,990700.0,0,3592,4,52,1,11,0,20000,210000,24,...,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0190,0,2016,1,53,1,1,4,0,1,0
4,2016-01-01,Maharashtra,Mumbai,Bandra Hub,1,LG Electronics 1,LG,Electronics,59410,15,50498.5,30,25,25,1262462.5,0,3592,4,52,1,11,0,20000,210000,24,...,0.0,80,Sunny,1.0,1,0,1,2,18,student,0,1,0,604,1.0216,0,2016,1,53,1,1,4,0,1,0


In [8]:
india_holidays = holidays.India(years = range(2015, 2026))

In [9]:
for date, name in india_holidays.items():
    print(date, name)

2016-01-26 Republic Day
2016-08-15 Independence Day
2016-10-02 Gandhi Jayanti
2016-05-21 Buddha Purnima
2016-10-30 Diwali
2016-08-25 Janmashtami
2016-10-11 Dussehra
2016-04-20 Mahavir Jayanti
2016-03-07 Maha Shivaratri
2016-11-14 Guru Nanak Jayanti
2016-10-12 Ashura
2016-12-13 Prophet's Birthday
2016-07-07 Eid al-Fitr
2016-09-13 Eid al-Adha
2016-03-25 Good Friday
2016-12-25 Christmas
2017-01-26 Republic Day
2017-08-15 Independence Day; Janmashtami
2017-10-02 Gandhi Jayanti
2017-05-10 Buddha Purnima
2017-10-19 Diwali
2017-09-30 Dussehra
2017-04-09 Mahavir Jayanti
2017-02-24 Maha Shivaratri
2017-11-04 Guru Nanak Jayanti
2017-10-01 Ashura
2017-12-02 Prophet's Birthday
2017-06-26 Eid al-Fitr
2017-09-02 Eid al-Adha
2017-04-14 Good Friday
2017-12-25 Christmas
2018-01-26 Republic Day
2018-08-15 Independence Day
2018-10-02 Gandhi Jayanti
2018-04-30 Buddha Purnima
2018-11-07 Diwali
2018-09-03 Janmashtami
2018-10-19 Dussehra
2018-03-29 Mahavir Jayanti
2018-02-13 Maha Shivaratri
2018-11-23 Guru N

In [10]:
df['is_holiday'] = df['date'].dt.date.isin(india_holidays.keys()).astype(int)

In [11]:
from scipy import stats

holiday_sales = df[df['is_holiday'] == 1]['sales']
non_holiday_sales = df[df['is_holiday'] == 0]['sales']

# T-test
t_stat, p_value = stats.ttest_ind(holiday_sales, non_holiday_sales, equal_var=False)
print(f"T-statistic: {t_stat}, P-value: {p_value}")

T-statistic: 19.5858731724712, P-value: 2.737659959474469e-85


In [12]:
print("Holiday Count:", len(holiday_sales))
print("Non-Holiday Count:", len(non_holiday_sales))

Holiday Count: 117078
Non-Holiday Count: 2695758


In [13]:
print("Holiday Avg:", holiday_sales.mean())
print("Non-Holiday Avg:", non_holiday_sales.mean())
print("Difference:", holiday_sales.mean() - non_holiday_sales.mean())

Holiday Avg: 319686.62098515517
Non-Holiday Avg: 288532.6004753208
Difference: 31154.020509834343


In [14]:
mean_diff = holiday_sales.mean() - non_holiday_sales.mean()

pooled_std = np.sqrt(
    (holiday_sales.std()**2 + non_holiday_sales.std()**2) / 2
)

cohens_d = mean_diff / pooled_std

print("Cohen's d:", cohens_d)

Cohen's d: 0.06096656623885414
